## Circuit Data transform Bronze to Silver

In [0]:
%run ../00-common/01-Environment_Config

In [0]:
%run ../00-common/02-Bronze_Helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.circuits"
silver_table = f"{catalog_name}.{silver_schema}.circuits"

In [0]:
# To use reader api options 
circuits_df = spark.read.table(bronze_table)

# circuits_df = spark.table(bronze_table) - without any reader options
# Both returns the same results

In [0]:
from pyspark.sql import functions as f

In [0]:
circuits_selected_df = circuits_df.select(
    f.col("circuitId"),
    f.col("circuitName"),
    f.col("lat"),
    f.col("long"),
    f.col("locality"),
    f.col("country"),
    f.col("ingestion_timestamp"),
    f.col("source_file")
)

In [0]:
# For column rename we can use ALIAS funcrion or also the below function

column_rename = {
    "circuitId" : "circuit_id",
    "circuitName" : "circuit_name",
    "lat" : "latitude",
    "long" : "longitude"
    }

circuit_renamed_df = circuits_selected_df.withColumnsRenamed(column_rename)

In [0]:
# To Drop duplicates and Null values in a column - distinct(), dropDuplicates() & filter()

# circuits_valid_df = circuit_renamed_df.distinct().filter(f.col("circuit_id").isNotNull())

circuits_valid_df = circuit_renamed_df.dropDuplicates(["circuit_id"]).filter(f.col("circuit_id").isNotNull())

In [0]:
circuits_final_df = circuits_valid_df.withColumn("circuit_name", f.initcap("circuit_name"))\
                                     . withColumn("locality", f.initcap("locality"))

In [0]:
circuits_final_df.write.format('delta').mode('overwrite').saveAsTable(silver_table)